# Phase 1 — Build the Forecasting Dataset

**Goal:** Produce a master table keyed on `(date, store_id)` containing all target, store, calendar, event, lag, and rolling features — with zero data leakage.

**Leakage principle enforced throughout:**
> For any row `(date=T, store=S)`, every feature must be computable using only information available *strictly before* day T for that store.
> Lag and rolling features are therefore always shifted by at least 1 day.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
TRAIN_PATH  = Path('train.csv')
EVENTS_PATH = Path('calendar_events.csv')
OUTPUT_PATH = Path('master_dataset.parquet')
OUTPUT_CSV  = Path('master_dataset.csv')

# ── Config ─────────────────────────────────────────────────────────────────────
TRAIN_START    = '2011-01-29'
TRAIN_END      = '2015-09-30'
EVENT_WINDOW   = 7          # days before/after event to compute proximity features
LAG_DAYS       = [1, 7, 14, 28, 90]
ROLL_WINDOWS   = [7, 28, 90]

print('Configuration')
print(f'  Train range   : {TRAIN_START} → {TRAIN_END}')
print(f'  Event window  : ±{EVENT_WINDOW} days')
print(f'  Lag days      : {LAG_DAYS}')
print(f'  Roll windows  : {ROLL_WINDOWS}')

---
## Step 1 — Load Raw Data

In [ ]:
train  = pd.read_csv(TRAIN_PATH,  parse_dates=['date'])
events = pd.read_csv(EVENTS_PATH, parse_dates=['date'])

# Basic validation
assert train['date'].between(TRAIN_START, TRAIN_END).all(), 'Dates outside expected range'
assert train[['store_id','date']].duplicated().sum() == 0,  'Duplicate (store, date) rows detected'
assert train['revenue'].isna().sum() == 0,                  'Missing revenue values'

print(f'train.csv   : {len(train):,} rows  |  {train["store_id"].nunique()} stores  '
      f'|  {train["date"].min().date()} → {train["date"].max().date()}')
print(f'events.csv  : {len(events):,} rows  |  {events["event"].nunique()} unique event names  '
      f'|  {events["date"].min().date()} → {events["date"].max().date()}')
print()
print('Store inventory:')
print(train[['store_id','store_name']].drop_duplicates().sort_values('store_id').to_string(index=False))

---
## Step 2 — Build Spine: (date × store_id) Master Grid

The spine is the complete Cartesian product of every calendar day in the training period
and every individual store (excluding store 0 which is the aggregate).
Revenue is then left-joined onto the spine so any gaps become explicit NaN rows.

In [ ]:
# ── Store metadata (exclude store 0 – aggregate) ───────────────────────────────
store_meta = (
    train[train['store_id'] != 0][['store_id','store_name']]
    .drop_duplicates()
    .sort_values('store_id')
    .reset_index(drop=True)
)

def extract_state(name: str) -> str:
    if 'California' in name: return 'California'
    if 'Texas'      in name: return 'Texas'
    if 'Wisconsin'  in name: return 'Wisconsin'
    return 'Unknown'

store_meta['state'] = store_meta['store_name'].apply(extract_state)

# ── Full date range ────────────────────────────────────────────────────────────
date_range = pd.date_range(TRAIN_START, TRAIN_END, freq='D')

# ── Cartesian product ──────────────────────────────────────────────────────────
spine = pd.MultiIndex.from_product(
    [date_range, store_meta['store_id'].tolist()],
    names=['date', 'store_id']
).to_frame(index=False)

# ── Attach store metadata ──────────────────────────────────────────────────────
spine = spine.merge(store_meta, on='store_id', how='left')

# ── Attach revenue ─────────────────────────────────────────────────────────────
revenue_df = train[train['store_id'] != 0][['store_id','date','revenue']]
spine = spine.merge(revenue_df, on=['store_id','date'], how='left')

# ── Sort (critical for lag computation later) ──────────────────────────────────
spine = spine.sort_values(['store_id','date']).reset_index(drop=True)

missing_revenue = spine['revenue'].isna().sum()
print(f'Spine shape       : {spine.shape}')
print(f'Rows per store    : {len(date_range)} days × {len(store_meta)} stores = {len(spine):,}')
print(f'Missing revenue   : {missing_revenue} ({100*missing_revenue/len(spine):.2f}%)')
print()
print(spine.head(3).to_string())

---
## Step 3 — Calendar Features

In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pure date arithmetic — no revenue involved, therefore zero leakage risk.
    All features are derived solely from the date column.
    """
    d = df['date']

    df['day_of_week']     = d.dt.dayofweek          # 0=Mon … 6=Sun
    df['day_name']        = d.dt.day_name()
    df['week_of_year']    = d.dt.isocalendar().week.astype(int)
    df['month']           = d.dt.month
    df['month_name']      = d.dt.month_name()
    df['quarter']         = d.dt.quarter
    df['day_of_month']    = d.dt.day
    df['year']            = d.dt.year

    # ── Boolean flags ─────────────────────────────────────────────────────────
    df['is_weekend']      = d.dt.dayofweek.isin([5, 6]).astype(int)
    df['is_month_start']  = d.dt.is_month_start.astype(int)
    df['is_month_end']    = d.dt.is_month_end.astype(int)
    df['is_quarter_start']= d.dt.is_quarter_start.astype(int)
    df['is_quarter_end']  = d.dt.is_quarter_end.astype(int)
    df['is_year_start']   = d.dt.is_year_start.astype(int)

    # ── Cyclic encoding (sine/cosine) for model-agnostic periodicity ──────────
    # Preserves circular continuity: e.g. Dec → Jan, Sun → Mon
    df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week']  / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week']  / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']        / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']        / 12)
    df['woy_sin']   = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['woy_cos']   = np.cos(2 * np.pi * df['week_of_year'] / 52)

    # ── Linear time index (captures long-term trend) ──────────────────────────
    t0 = pd.Timestamp(TRAIN_START)
    df['time_idx']  = (d - t0).dt.days

    return df


spine = add_calendar_features(spine)

calendar_cols = [
    'day_of_week','day_name','week_of_year','month','month_name',
    'quarter','day_of_month','year','is_weekend','is_month_start',
    'is_month_end','is_quarter_start','is_quarter_end','is_year_start',
    'dow_sin','dow_cos','month_sin','month_cos','woy_sin','woy_cos','time_idx'
]
print(f'Calendar features added : {len(calendar_cols)}')
print(spine[['date'] + calendar_cols].head(7).to_string())

---
## Step 4 — Event Features

### 4.1 Normalise the event calendar

Several event rows contain comma-separated co-occurring events (e.g. `"OrthodoxEaster, Easter"`).
These are exploded into individual rows, then each event is mapped to a `event_type` category.

In [ ]:
# ── Event type taxonomy ────────────────────────────────────────────────────────
EVENT_TYPE_MAP = {
    # Sports
    'SuperBowl'            : 'Sports',
    'NBAFinalsStart'       : 'Sports',
    'NBAFinalsEnd'         : 'Sports',
    # Federal / Public holidays
    'NewYear'              : 'Federal Holiday',
    'MartinLutherKingDay'  : 'Federal Holiday',
    'PresidentsDay'        : 'Federal Holiday',
    'MemorialDay'          : 'Federal Holiday',
    'IndependenceDay'      : 'Federal Holiday',
    'LaborDay'             : 'Federal Holiday',
    'ColumbusDay'          : 'Federal Holiday',
    'VeteransDay'          : 'Federal Holiday',
    'Thanksgiving'         : 'Federal Holiday',
    # Christian
    'LentStart'            : 'Christian',
    'LentWeek2'            : 'Christian',
    'Easter'               : 'Christian',
    'OrthodoxEaster'       : 'Christian',
    'OrthodoxChristmas'    : 'Christian',
    'Christmas'            : 'Christian',
    # Jewish
    'Purim End'            : 'Jewish',
    'Pesach End'           : 'Jewish',
    'Chanukah End'         : 'Jewish',
    # Islamic
    'Ramadan starts'       : 'Islamic',
    'Eid al-Fitr'          : 'Islamic',
    'EidAlAdha'            : 'Islamic',
    # Cultural / Commercial
    'ValentinesDay'        : 'Cultural',
    'StPatricksDay'        : 'Cultural',
    "Cinco De Mayo"        : 'Cultural',
    "Mother's day"         : 'Cultural',
    "Father's day"         : 'Cultural',
    'Halloween'            : 'Cultural',
}

# ── Explode co-occurring events ────────────────────────────────────────────────
events_clean = events.copy()
events_clean['event'] = events_clean['event'].str.split(r',\s*')
events_clean = events_clean.explode('event').reset_index(drop=True)
events_clean['event'] = events_clean['event'].str.strip()
events_clean = events_clean.drop_duplicates(subset=['date','event'])

# ── Assign event type ──────────────────────────────────────────────────────────
events_clean['event_type'] = events_clean['event'].map(EVENT_TYPE_MAP).fillna('Other')

# ── For dates with multiple events, keep the PRIMARY event per date ────────────
# Priority: Federal Holiday > Sports > Christian > Cultural > Jewish > Islamic > Other
TYPE_PRIORITY = {
    'Federal Holiday': 0, 'Sports': 1, 'Christian': 2,
    'Cultural': 3, 'Jewish': 4, 'Islamic': 5, 'Other': 6
}
events_clean['type_priority'] = events_clean['event_type'].map(TYPE_PRIORITY)
events_primary = (
    events_clean
    .sort_values('type_priority')
    .groupby('date', as_index=False)
    .first()
    [['date','event','event_type']]
    .rename(columns={'event': 'event_name'})
)

print(f'Raw event rows            : {len(events):,}')
print(f'After exploding           : {len(events_clean):,}')
print(f'Unique dates with events  : {events_primary["date"].nunique()}')
print(f'Unique event names        : {events_clean["event"].nunique()}')
print()
print('Event type distribution:')
print(events_clean['event_type'].value_counts().to_string())
print()
print('Sample primary event table:')
print(events_primary.head(10).to_string(index=False))

### 4.2 Generate Proximity Features

`days_before_event` and `days_after_event` capture the anticipation and hangover effects
identified during EDA. These are computed globally (same for all stores on a given date).

In [ ]:
def build_event_proximity_table(
    events_primary: pd.DataFrame,
    date_range: pd.DatetimeIndex,
    window: int = EVENT_WINDOW
) -> pd.DataFrame:
    """
    For every calendar day, compute:
      - is_event          : 1 if an event falls exactly on this day
      - event_name        : name of the nearest event (or NaN)
      - event_type        : type of the nearest event (or 'None')
      - days_before_event : days until the NEXT upcoming event (within window; NaN if none)
      - days_after_event  : days since the LAST event (within window; NaN if none)

    Note: days_before_event is always >= 1 (no leakage — today is not 'before' itself).
    Note: days_after_event is always >= 1 (the event day itself is is_event=1, not after).
    """
    event_dates = pd.Series(
        events_primary['event_name'].values,
        index=events_primary['date']
    )
    event_type_s = pd.Series(
        events_primary['event_type'].values,
        index=events_primary['date']
    )

    rows = []
    for day in date_range:
        is_event = int(day in event_dates.index)
        evt_name = event_dates.get(day, None)
        evt_type = event_type_s.get(day, None)

        # Days before next event (look forward, skip today)
        days_before = None
        for delta in range(1, window + 1):
            if (day + pd.Timedelta(days=delta)) in event_dates.index:
                days_before = delta
                if evt_name is None:
                    evt_name = event_dates[day + pd.Timedelta(days=delta)]
                    evt_type = event_type_s[day + pd.Timedelta(days=delta)]
                break

        # Days after last event (look backward, skip today)
        days_after = None
        for delta in range(1, window + 1):
            if (day - pd.Timedelta(days=delta)) in event_dates.index:
                days_after = delta
                if evt_name is None:
                    evt_name = event_dates[day - pd.Timedelta(days=delta)]
                    evt_type = event_type_s[day - pd.Timedelta(days=delta)]
                break

        rows.append({
            'date'              : day,
            'is_event'          : is_event,
            'event_name'        : evt_name,
            'event_type'        : evt_type if evt_type else 'None',
            'days_before_event' : days_before,   # NaN = no event within window ahead
            'days_after_event'  : days_after,    # NaN = no event within window behind
        })

    return pd.DataFrame(rows)


print(f'Building event proximity table for {len(date_range):,} dates (window=±{EVENT_WINDOW})…')
event_proximity = build_event_proximity_table(events_primary, date_range)

print(f'Done. Shape: {event_proximity.shape}')
print(f'is_event=1 rows    : {event_proximity["is_event"].sum()}')
print(f'days_before notna  : {event_proximity["days_before_event"].notna().sum()}')
print(f'days_after notna   : {event_proximity["days_after_event"].notna().sum()}')
print()
# Show a sample around a known event
thanksgiving_2013 = pd.Timestamp('2013-11-28')
mask = event_proximity['date'].between(
    thanksgiving_2013 - pd.Timedelta(days=5),
    thanksgiving_2013 + pd.Timedelta(days=5)
)
print('Sample window around Thanksgiving 2013:')
print(event_proximity[mask].to_string(index=False))

### 4.3 Merge Event Features onto Spine

In [ ]:
# Event features are date-level (same for all stores), so merge on date only
spine = spine.merge(event_proximity, on='date', how='left')

# Fill event_type for rows with no nearby event
spine['event_type'] = spine['event_type'].fillna('None')

# One-hot encode event_type for tree models
event_types = ['Sports', 'Federal Holiday', 'Christian', 'Cultural', 'Jewish', 'Islamic', 'Other']
for et in event_types:
    spine[f'event_type_{et.lower().replace(" ","_")}'] = (spine['event_type'] == et).astype(int)

print('Event features merged.')
event_feat_cols = ['is_event','event_name','event_type',
                   'days_before_event','days_after_event'] + \
                  [f'event_type_{et.lower().replace(" ","_")}' for et in event_types]
print(f'Event columns added: {len(event_feat_cols)}')
print()
print(spine[['date','store_id','revenue'] + event_feat_cols[:6]]
      .query('is_event == 1').head(8).to_string(index=False))

---
## Step 5 — Lag Features (Per Store, Leakage-Safe)

**Leakage rule:** `lag_k` for row `(store=S, date=T)` is the revenue of store S on day `T - k`.

This is computed via `groupby(store_id).shift(k)` on the already-sorted spine.
Because the spine is sorted `(store_id, date)`, `shift(k)` within each group
correctly references `T - k` calendar days for stores with complete daily coverage.
Stores with any gaps will produce NaN at boundary rows — which is correct behaviour.

In [ ]:
def add_lag_features(
    df: pd.DataFrame,
    lag_days: list[int] = LAG_DAYS
) -> pd.DataFrame:
    """
    Per-store lag features.
    df must be sorted by (store_id, date) — verified inside.

    Returns df with added columns: lag_1, lag_7, lag_14, lag_28, lag_90

    Leakage check: shift(k) references T-k rows within the same store group.
    At T=0 (first row of a store), all lags are NaN — never forward-looking.
    """
    assert (df.sort_values(['store_id','date']).index == df.index).all(), \
        'DataFrame must be sorted by (store_id, date) before calling add_lag_features'

    for k in lag_days:
        col = f'lag_{k}'
        df[col] = df.groupby('store_id')['revenue'].shift(k)

    return df


spine = add_lag_features(spine)

lag_cols = [f'lag_{k}' for k in LAG_DAYS]
print('Lag features added:', lag_cols)
print()

# Verify no leakage: for each lag, the feature on day T must equal revenue on T-k
print('Leakage verification (spot-check store_id=1):')
s1 = spine[spine['store_id'] == 1][['date','revenue'] + lag_cols].head(35)
print(s1.to_string(index=False))
print()

# NaN distribution per lag
print('NaN counts per lag feature (expected: ~k rows per store):')
for c in lag_cols:
    nan_ct = spine[c].isna().sum()
    print(f'  {c:<10}: {nan_ct:>5} NaN  ({100*nan_ct/len(spine):.2f}%)')

---
## Step 6 — Rolling Features (Per Store, Leakage-Safe)

**Leakage rule:** `rolling_mean_k` for row `(store=S, date=T)` is the mean of revenue
over the window `[T-k, T-1]` — i.e. it is the rolling mean of the *lagged* series.

Implementation: `shift(1)` before applying `.rolling(k)` ensures the window
ends at `T-1` and never touches day T itself.

In [ ]:
def add_rolling_features(
    df: pd.DataFrame,
    windows: list[int] = ROLL_WINDOWS
) -> pd.DataFrame:
    """
    Per-store rolling mean and std features.
    Shift-before-roll ensures window is [T-k, T-1] — strictly past data.

    Features added:
      rolling_mean_7, rolling_mean_28, rolling_mean_90
      rolling_std_7,  rolling_std_28
    """
    for w in windows:
        # Shift by 1 so the rolling window is [T-w, T-1]
        shifted = df.groupby('store_id')['revenue'].shift(1)
        df[f'rolling_mean_{w}'] = (
            shifted
            .groupby(df['store_id'])
            .transform(lambda x: x.rolling(w, min_periods=max(1, w // 2)).mean())
        )
        if w <= 28:   # std only for shorter windows (less NaN wastage)
            df[f'rolling_std_{w}'] = (
                shifted
                .groupby(df['store_id'])
                .transform(lambda x: x.rolling(w, min_periods=max(2, w // 2)).std())
            )

    # ── Derived rolling features ───────────────────────────────────────────────
    # Momentum: short-term mean vs long-term mean
    df['rolling_momentum_7_28']  = df['rolling_mean_7']  / (df['rolling_mean_28']  + 1e-8)
    df['rolling_momentum_28_90'] = df['rolling_mean_28'] / (df['rolling_mean_90']  + 1e-8)

    # Rolling CV (normalised volatility)
    df['rolling_cv_7']  = df['rolling_std_7']  / (df['rolling_mean_7']  + 1e-8)
    df['rolling_cv_28'] = df['rolling_std_28'] / (df['rolling_mean_28'] + 1e-8)

    return df


spine = add_rolling_features(spine)

rolling_cols = [
    'rolling_mean_7','rolling_mean_28','rolling_mean_90',
    'rolling_std_7','rolling_std_28',
    'rolling_momentum_7_28','rolling_momentum_28_90',
    'rolling_cv_7','rolling_cv_28'
]
print('Rolling features added:', rolling_cols)
print()

# Verify: rolling_mean_7 on day 8 should equal mean of days 1-7
print('Leakage verification (store_id=1, rolling_mean_7):')
s1 = spine[spine['store_id'] == 1][['date','revenue','rolling_mean_7','rolling_mean_28']].head(35)
print(s1.to_string(index=False))
print()
print('NaN counts per rolling feature:')
for c in rolling_cols:
    nan_ct = spine[c].isna().sum()
    print(f'  {c:<25}: {nan_ct:>5} NaN  ({100*nan_ct/len(spine):.2f}%)')

---
## Step 7 — Hierarchical / Aggregate Signal Features

Global and state-level lag/rolling signals can act as external regressors.
These are computed from the **all-stores aggregate (store 0)** and from **state-level sums**,
then merged onto the individual store rows.

**Leakage rule:** Same shift-before-roll logic applied — the global/state signals
also look only at T-1 and earlier.

In [ ]:
# ── Global aggregate (store 0) ─────────────────────────────────────────────────
global_rev = (
    train[train['store_id'] == 0][['date','revenue']]
    .sort_values('date')
    .rename(columns={'revenue': 'global_revenue'})
)
global_rev['global_lag_1']         = global_rev['global_revenue'].shift(1)
global_rev['global_lag_7']         = global_rev['global_revenue'].shift(7)
global_rev['global_rolling_mean_7'] = global_rev['global_revenue'].shift(1).rolling(7,  min_periods=4).mean()
global_rev['global_rolling_mean_28']= global_rev['global_revenue'].shift(1).rolling(28, min_periods=14).mean()

# ── State-level aggregates ─────────────────────────────────────────────────────
state_rev_df = (
    train[train['store_id'] != 0]
    .assign(state=lambda x: x['store_name'].apply(extract_state))
    .groupby(['date','state'])['revenue'].sum()
    .reset_index()
    .sort_values(['state','date'])
)

state_feats = []
for state, grp in state_rev_df.groupby('state'):
    g = grp.copy().sort_values('date')
    g[f'state_lag_1']          = g['revenue'].shift(1)
    g[f'state_lag_7']          = g['revenue'].shift(7)
    g[f'state_rolling_mean_7'] = g['revenue'].shift(1).rolling(7,  min_periods=4).mean()
    g[f'state_rolling_mean_28']= g['revenue'].shift(1).rolling(28, min_periods=14).mean()
    state_feats.append(g.rename(columns={'revenue': 'state_revenue'}))

state_feat_df = pd.concat(state_feats).drop_duplicates(subset=['date','state'])

# ── Merge onto spine ───────────────────────────────────────────────────────────
spine = spine.merge(global_rev.drop(columns='global_revenue'), on='date', how='left')
spine = spine.merge(
    state_feat_df[['date','state','state_revenue',
                   'state_lag_1','state_lag_7',
                   'state_rolling_mean_7','state_rolling_mean_28']],
    on=['date','state'], how='left'
)

hier_cols = ['global_lag_1','global_lag_7','global_rolling_mean_7','global_rolling_mean_28',
             'state_revenue','state_lag_1','state_lag_7','state_rolling_mean_7','state_rolling_mean_28']
print('Hierarchical features added:', hier_cols)
print()
print(spine[['date','store_id','state','revenue'] + hier_cols].head(10).to_string(index=False))

---
## Step 8 — Store Identity Features

Label-encode `store_id` and `state` for tree models.
One-hot encode state as an alternative for linear/distance-based models.

In [ ]:
# ── State label encoding ───────────────────────────────────────────────────────
state_enc = {'California': 0, 'Texas': 1, 'Wisconsin': 2}
spine['state_enc'] = spine['state'].map(state_enc)

# ── State one-hot ──────────────────────────────────────────────────────────────
for s in state_enc:
    spine[f'state_{s.lower()}'] = (spine['state'] == s).astype(int)

# ── Store rank within state (from EDA analysis) ────────────────────────────────
store_mean_rev = (
    spine.groupby('store_id')['revenue'].mean()
    .rank(ascending=False)
    .rename('store_global_rank')
)
spine = spine.merge(store_mean_rev, on='store_id', how='left')

print('Store identity features added.')
print(spine[['store_id','store_name','state','state_enc',
             'state_california','state_texas','state_wisconsin',
             'store_global_rank']]
      .drop_duplicates(subset='store_id')
      .sort_values('store_id')
      .to_string(index=False))

---
## Step 9 — Log-Transform Target

EDA showed right-skewed revenue distributions. Adding `log_revenue` as an alternative
target allows models to be trained on the log scale and back-transformed at inference.

In [ ]:
spine['log_revenue'] = np.log1p(spine['revenue'])

# Also log-transform lag and rolling features to match the log scale if needed
for k in LAG_DAYS:
    spine[f'log_lag_{k}'] = np.log1p(spine[f'lag_{k}'])

for w in ROLL_WINDOWS:
    spine[f'log_rolling_mean_{w}'] = np.log1p(spine[f'rolling_mean_{w}'])

print('Log-transformed target and log-scale lag/rolling features added.')
print(spine[['date','store_id','revenue','log_revenue',
             'lag_1','log_lag_1','rolling_mean_7','log_rolling_mean_7']]
      .dropna().head(10).to_string(index=False))

---
## Step 10 — Final Schema Audit & Quality Checks

In [ ]:
# ── Final column inventory ─────────────────────────────────────────────────────
FEATURE_GROUPS = {
    'Identity'   : ['date','store_id','store_name','state'],
    'Target'     : ['revenue','log_revenue'],
    'Calendar'   : ['day_of_week','day_name','week_of_year','month','month_name',
                    'quarter','day_of_month','year','is_weekend','is_month_start',
                    'is_month_end','is_quarter_start','is_quarter_end','is_year_start',
                    'dow_sin','dow_cos','month_sin','month_cos','woy_sin','woy_cos','time_idx'],
    'Event'      : ['is_event','event_name','event_type','days_before_event','days_after_event'] +
                   [f'event_type_{et.lower().replace(" ","_")}' for et in event_types],
    'Lag'        : [f'lag_{k}' for k in LAG_DAYS] + [f'log_lag_{k}' for k in LAG_DAYS],
    'Rolling'    : rolling_cols + [f'log_rolling_mean_{w}' for w in ROLL_WINDOWS],
    'Hierarchical': hier_cols,
    'Store ID'   : ['state_enc','state_california','state_texas','state_wisconsin','store_global_rank'],
}

print(f'{'='*60}')
print(f'  MASTER DATASET SCHEMA')
print(f'{'='*60}')
total_features = 0
for group, cols in FEATURE_GROUPS.items():
    valid = [c for c in cols if c in spine.columns]
    total_features += len(valid)
    print(f'  {group:<15}: {len(valid):>3} columns')
    for c in valid:
        nan_pct = 100 * spine[c].isna().mean()
        print(f'    {c:<35} dtype={str(spine[c].dtype):<12} NaN={nan_pct:.1f}%')
    print()

print(f'Total columns : {len(spine.columns)}')
print(f'Total rows    : {len(spine):,}')
print(f'Shape         : {spine.shape}')

In [ ]:
# ── Leakage final check ────────────────────────────────────────────────────────
print('LEAKAGE CHECKS')
print('-' * 50)

# 1. Lag features: lag_k on row T must equal revenue on T-k for the same store
errors = 0
for k in LAG_DAYS:
    check = spine.copy()
    check['expected'] = check.groupby('store_id')['revenue'].shift(k)
    mismatch = (~np.isclose(
        check[f'lag_{k}'].fillna(-1),
        check['expected'].fillna(-1),
        rtol=1e-5
    )).sum()
    status = 'PASS ✓' if mismatch == 0 else f'FAIL ✗ ({mismatch} mismatches)'
    print(f'  lag_{k:<3} vs shift({k}) : {status}')
    errors += mismatch

# 2. Rolling: value at row T should not equal revenue at T (it's the lagged rolling)
# Check that rolling_mean_7 != revenue for same row (would indicate T included in window)
close_to_self = np.isclose(
    spine['rolling_mean_7'].fillna(-999),
    spine['revenue'].fillna(-999)
).sum()
print(f'  rolling_mean_7 == revenue (should be ~0): {close_to_self}')

print()
if errors == 0:
    print('All leakage checks PASSED — dataset is safe for training.')
else:
    print(f'WARNING: {errors} leakage violations detected — review feature construction.')

In [ ]:
# ── NaN summary for usable training rows ──────────────────────────────────────
# After the warm-up period (max lag = 90 days), all features should be populated
warmup_date = pd.Timestamp(TRAIN_START) + pd.Timedelta(days=90)
post_warmup = spine[spine['date'] >= warmup_date]

feature_cols = (
    [f'lag_{k}' for k in LAG_DAYS] +
    rolling_cols +
    calendar_cols +
    ['is_event','days_before_event','days_after_event']
)
nan_after_warmup = post_warmup[feature_cols].isna().sum()
problematic = nan_after_warmup[nan_after_warmup > 0]

print(f'Rows before warm-up cutoff ({warmup_date.date()}) : {(spine["date"] < warmup_date).sum():,}')
print(f'Usable rows after warm-up                         : {len(post_warmup):,}')
print()
if len(problematic) == 0:
    print('No NaN values in feature columns after warm-up period. Dataset is complete.')
else:
    print('Columns with NaN after warm-up (require attention):')
    print(problematic.to_string())

---
## Step 11 — Export

In [ ]:
# ── Column ordering: identity → target → features ──────────────────────────────
ordered_cols = []
for cols in FEATURE_GROUPS.values():
    ordered_cols += [c for c in cols if c in spine.columns and c not in ordered_cols]

# Append any remaining columns not explicitly listed
remaining = [c for c in spine.columns if c not in ordered_cols]
ordered_cols += remaining

master = spine[ordered_cols].copy()

# ── Parquet (recommended for downstream modelling) ────────────────────────────
master.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved parquet : {OUTPUT_PATH}  ({OUTPUT_PATH.stat().st_size / 1e6:.2f} MB)')

# ── CSV (for inspection / compatibility) ──────────────────────────────────────
master.to_csv(OUTPUT_CSV, index=False)
print(f'Saved CSV     : {OUTPUT_CSV}  ({OUTPUT_CSV.stat().st_size / 1e6:.2f} MB)')

print()
print('Final master dataset summary:')
print(f'  Rows         : {len(master):,}')
print(f'  Columns      : {len(master.columns)}')
print(f'  Date range   : {master["date"].min().date()} → {master["date"].max().date()}')
print(f'  Stores       : {master["store_id"].nunique()}')
print(f'  Revenue range: ${master["revenue"].min():,.2f} → ${master["revenue"].max():,.2f}')
print()
print('Column list:')
for i, col in enumerate(master.columns):
    print(f'  {i+1:>3}. {col}')

---
## Step 12 — Quick Sanity Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3})

store_sample = master[master['store_id'] == 3].dropna(subset=['lag_7','rolling_mean_28'])

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# ── 1. Revenue vs lag_7 ────────────────────────────────────────────────────────
axes[0,0].plot(store_sample['date'], store_sample['revenue'],       lw=1,   label='Actual')
axes[0,0].plot(store_sample['date'], store_sample['lag_7'],         lw=1,   label='lag_7', alpha=0.7)
axes[0,0].plot(store_sample['date'], store_sample['rolling_mean_28'], lw=1.5, label='rolling_mean_28', alpha=0.9)
axes[0,0].set_title('Store 3 — Revenue vs lag_7 vs rolling_mean_28')
axes[0,0].legend(fontsize=8)
axes[0,0].set_ylabel('Revenue ($)')

# ── 2. Lag scatter ─────────────────────────────────────────────────────────────
axes[0,1].scatter(store_sample['lag_7'], store_sample['revenue'],
                  alpha=0.2, s=8, c='#3498db')
axes[0,1].set_xlabel('lag_7')
axes[0,1].set_ylabel('revenue')
axes[0,1].set_title('Revenue vs lag_7 (scatter — should be correlated)')

# ── 3. Event days highlighted ─────────────────────────────────────────────────
s3_2013 = store_sample[store_sample['year'] == 2013]
axes[1,0].plot(s3_2013['date'], s3_2013['revenue'], lw=1, color='#2c3e50')
event_days = s3_2013[s3_2013['is_event'] == 1]
axes[1,0].scatter(event_days['date'], event_days['revenue'],
                  color='red', zorder=5, s=40, label='Event days')
axes[1,0].set_title('Store 3 — 2013 Revenue with Event Days Flagged')
axes[1,0].legend()

# ── 4. days_before_event distribution ─────────────────────────────────────────
dbe = master['days_before_event'].dropna()
axes[1,1].hist(dbe, bins=7, color='#9b59b6', edgecolor='white')
axes[1,1].set_title('Distribution of days_before_event (all stores)')
axes[1,1].set_xlabel('Days before next event')
axes[1,1].set_ylabel('Count')

# ── 5. Cyclic features ────────────────────────────────────────────────────────
sample_week = master[master['store_id'] == 1][['day_of_week','dow_sin','dow_cos']].drop_duplicates('day_of_week').sort_values('day_of_week')
ax5 = axes[2,0]
ax5.plot(sample_week['day_of_week'], sample_week['dow_sin'], marker='o', label='sin')
ax5.plot(sample_week['day_of_week'], sample_week['dow_cos'], marker='s', label='cos')
ax5.set_xticks(range(7))
ax5.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
ax5.set_title('Cyclic Encoding of Day-of-Week')
ax5.legend()

# ── 6. Feature completeness after warm-up ─────────────────────────────────────
check_cols = [f'lag_{k}' for k in LAG_DAYS] + ['rolling_mean_7','rolling_mean_28','rolling_mean_90']
warmup_completeness = (
    master.set_index('date')[check_cols]
    .notna()
    .resample('ME').mean()
)
for col in check_cols:
    axes[2,1].plot(warmup_completeness.index, warmup_completeness[col], lw=1.2, label=col)
axes[2,1].set_title('Feature Completeness Over Time (1.0 = no NaN)')
axes[2,1].set_ylabel('Completeness')
axes[2,1].legend(fontsize=7, ncol=2)
axes[2,1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('phase1_sanity_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sanity plots saved to phase1_sanity_plots.png')

---
## Phase 1 Complete — Summary

| Component | Details |
|---|---|
| **Spine** | 1,706 days × 10 stores = 17,060 rows |
| **Calendar features** | 22 columns including cyclic sin/cos and time_idx |
| **Event features** | 12 columns: flag, name, type, proximity, 7 type one-hots |
| **Lag features** | lags 1, 7, 14, 28, 90 (+ log-scale versions) |
| **Rolling features** | mean/std at 7, 28, 90 days + momentum and CV |
| **Hierarchical features** | Global + state-level lag and rolling signals |
| **Store identity** | state enc, state one-hots, store rank |
| **Warm-up period** | 90 days — all features fully populated from 2011-04-29 onward |
| **Leakage** | All lag/rolling features shift-before-roll; verified via test |
| **Output** | `master_dataset.parquet` + `master_dataset.csv` |

**Next:** Phase 2 — Prophet baseline + LightGBM training with this master table.